# 6. Temporal-Difference Learning

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 6: Temporal-Difference Learning** (Sayfa 115-145)

## İçindekiler
1. TD Prediction *(s. 115-122)*
2. TD(0) Algoritması *(s. 119-120)*
3. SARSA (On-policy TD Control) *(s. 129-131)*
4. Q-Learning (Off-policy TD Control) *(s. 131-132)*
5. Expected SARSA *(s. 133-134)*
6. Maximization Bias ve Double Q-Learning *(s. 134-136)*

---
## 6.1 TD Learning Nedir?

📖 **Referans:** Sutton & Barto, Sayfa 115-119, Section 6.1

> *"TD learning is a combination of Monte Carlo ideas and dynamic programming (DP) ideas."* (s. 115)

**Temporal-Difference (TD)** learning, MC ve DP'nin en iyi özelliklerini birleştirir:

| Özellik | DP | MC | TD |
|---------|----|----|----|
| Model gerekli | Evet | Hayır | Hayır |
| Bootstrapping | Evet | Hayır | Evet |
| Episode bitmeli | Hayır | Evet | Hayır |

> *"Like Monte Carlo methods, TD methods can learn directly from raw experience without a model of the environment's dynamics. Like DP, TD methods update estimates based in part on other learned estimates, without waiting for a final outcome (they bootstrap)."* (s. 115)

### TD'nin Temel Fikri (s. 119)

**MC:** Episode sonunda tam return ile güncelle:
$$V(S_t) \leftarrow V(S_t) + \alpha [G_t - V(S_t)]$$

**TD:** Her adımda **tahmin** ile güncelle (bootstrapping) - Equation 6.2 (s. 119):
$$V(S_t) \leftarrow V(S_t) + \alpha [R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

### TD Error - Equation 6.5 (s. 121)

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

> *"This error, called the TD error, arises in various forms throughout reinforcement learning."* (s. 121)

In [ ]:
# Kod Örneği: Cliff Walking Environment
# Referans: Example 6.6 (s. 132) - Cliff Walking

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Tuple, List

class CliffWalkingEnv:
    """
    Cliff Walking Environment.
    Referans: Example 6.6 (s. 132)
    
    "This is a standard undiscounted, episodic task, with start and goal states,
    and the usual actions causing movement up, down, right, and left."
    
    4x12 grid. Start: bottom-left. Goal: bottom-right.
    Cliff: bottom row (except start and goal).
    "Reward is −1 on all transitions except those into the region marked 'The Cliff.'
    Stepping into this region incurs a reward of −100 and sends the agent instantly 
    back to the start."
    """
    
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)  # Bottom-left
        self.goal = (3, 11)  # Bottom-right
        self.cliff = [(3, i) for i in range(1, 11)]  # "The Cliff" (s. 132)
        
        self.n_states = self.rows * self.cols
        self.n_actions = 4  # up, right, down, left
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        
        self.reset()
    
    def reset(self):
        self.position = self.start
        return self.position
    
    def step(self, action):
        row, col = self.position
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        new_pos = (new_row, new_col)
        
        # "Stepping into this region incurs a reward of −100"
        if new_pos in self.cliff:
            self.position = self.start  # "sends the agent instantly back to the start"
            return self.position, -100, False
        
        self.position = new_pos
        
        if self.position == self.goal:
            return self.position, -1, True
        
        return self.position, -1, False  # "Reward is −1 on all transitions"

env = CliffWalkingEnv()
print(f"Start: {env.start}, Goal: {env.goal}")
print(f"Cliff positions: {env.cliff}")

In [ ]:
def visualize_cliff_world(env, V=None, Q=None, policy=None, path=None, title="Cliff Walking"):
    """Cliff walking environment'ı görselleştir."""
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Draw grid
    for row in range(env.rows):
        for col in range(env.cols):
            pos = (row, col)
            
            if pos == env.start:
                color = 'lightgreen'
                label = 'S'
            elif pos == env.goal:
                color = 'gold'
                label = 'G'
            elif pos in env.cliff:
                color = 'gray'
                label = 'C'
            else:
                color = 'white'
                label = ''
            
            rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                                 facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            if label:
                ax.text(col + 0.5, env.rows - row - 0.5, label,
                       ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Draw path if provided
    if path:
        path_x = [p[1] + 0.5 for p in path]
        path_y = [env.rows - p[0] - 0.5 for p in path]
        ax.plot(path_x, path_y, 'b-o', linewidth=2, markersize=4, alpha=0.7)
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

visualize_cliff_world(env)

---
## 6.2 TD(0) Prediction

📖 **Referans:** Sutton & Barto, Sayfa 119-120, Algorithm on page 120

> *"Because TD(0) bases its update in part on an existing estimate, we say that it is a bootstrapping method, like DP."* (s. 119)

Verilen policy için V(s) tahmin et.

### Tabular TD(0) Algorithm (s. 120)

```
Input: the policy π to be evaluated
Algorithm parameter: step size α ∈ (0, 1]
Initialize V(s), for all s ∈ S+, arbitrarily except that V(terminal) = 0

Loop for each episode:
    Initialize S
    Loop for each step of episode:
        A ← action given by π for S
        Take action A, observe R, S'
        V(S) ← V(S) + α[R + γV(S') - V(S)]
        S ← S'
    until S is terminal
```

> *"The target for the TD(0) update is $R_{t+1} + \gamma V(S_{t+1})$"* (s. 119)

In [ ]:
def td_prediction(env, policy, n_episodes=500, alpha=0.1, gamma=1.0):
    """
    TD(0) Prediction.
    Referans: Algorithm (s. 120) - "Tabular TD(0) for estimating v_π"
    
    Args:
        env: Environment
        policy: Function state -> action
        n_episodes: Number of episodes
        alpha: "step size α ∈ (0, 1]" (s. 120)
        gamma: Discount factor
    
    Returns:
        V: State value function
    """
    # "Initialize V(s), for all s ∈ S+, arbitrarily except that V(terminal) = 0"
    V = defaultdict(float)
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        state = env.reset()  # "Initialize S"
        
        # "Loop for each step of episode:"
        while True:
            action = policy(state)  # "A ← action given by π for S"
            next_state, reward, done = env.step(action)  # "Take action A, observe R, S'"
            
            # "V(S) ← V(S) + α[R + γV(S') - V(S)]" - Equation 6.2
            td_target = reward + gamma * V[next_state]
            td_error = td_target - V[state]
            V[state] += alpha * td_error
            
            if done:  # "until S is terminal"
                break
            
            state = next_state  # "S ← S'"
    
    return V

# Test with a simple policy
def simple_policy(state):
    row, col = state
    if col < 11:
        return 1  # right
    return 2  # down

V = td_prediction(env, simple_policy, n_episodes=1000)
print(f"TD(0) estimated values for {len(V)} states")

---
## 6.3 SARSA: On-policy TD Control

📖 **Referans:** Sutton & Barto, Sayfa 129-131, Section 6.4

> *"We turn now to the use of TD prediction methods for the control problem."* (s. 129)

**SARSA** = State-Action-Reward-State-Action

### Update Rule - Equation 6.7 (s. 130)

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

> *"This update is done after every transition from a nonterminal state $S_t$. If $S_{t+1}$ is terminal, then $Q(S_{t+1}, A_{t+1})$ is defined as zero."* (s. 130)

### Özellikler (s. 130)
- **On-policy**: Aynı ε-greedy policy'yi hem öğrenmek hem de explore etmek için kullanır
- > *"We use every element of the quintuple of events, $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$, that make up a transition from one state–action pair to the next. This quintuple gives rise to the name Sarsa for the algorithm."*

In [ ]:
def sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    SARSA (On-policy TD Control).
    Referans: Algorithm (s. 130) - "Sarsa: An on-policy TD control algorithm"
    """
    # "Initialize Q(s, a), for all s ∈ S+, a ∈ A(s), arbitrarily except that Q(terminal, ·) = 0"
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        """ε-greedy policy: "Choose A from S using policy derived from Q (e.g., ε-greedy)" """
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_rewards = []
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        state = env.reset()  # "Initialize S"
        action = epsilon_greedy(state)  # "Choose A from S using policy derived from Q"
        
        total_reward = 0
        
        # "Loop for each step of episode:"
        while True:
            # "Take action A, observe R, S'"
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # "Choose A' from S' using policy derived from Q"
            next_action = epsilon_greedy(next_state)
            
            # "Q(S,A) ← Q(S,A) + α[R + γQ(S',A') − Q(S,A)]" - Equation 6.7
            td_target = reward + gamma * Q[next_state][next_action]
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:  # "until S is terminal"
                break
            
            # "S ← S'; A ← A'"
            state = next_state
            action = next_action
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=500)
print(f"SARSA trained for {len(Q_sarsa)} states")

---
## 6.4 Q-Learning: Off-policy TD Control

📖 **Referans:** Sutton & Barto, Sayfa 131-132, Section 6.5

> *"One of the early breakthroughs in reinforcement learning was the development of an off-policy TD control algorithm known as Q-learning"* (s. 131)

### Update Rule - Equation 6.8 (s. 131)

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

> *"In this case, the learned action-value function, Q, directly approximates $q_*$, the optimal action-value function, independent of the policy being followed."* (s. 131)

### Özellikler (s. 131)
- **Off-policy**: Greedy policy'yi (target) öğrenirken ε-greedy (behavior) ile explore eder
- Target policy: $\pi(s) = \arg\max_a Q(s, a)$ (greedy)
- Behavior policy: ε-greedy

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Q-Learning (Off-policy TD Control).
    Referans: Algorithm (s. 131) - "Q-learning: An off-policy TD control algorithm"
    """
    # "Initialize Q(s, a), for all s ∈ S+, a ∈ A(s), arbitrarily except that Q(terminal, ·) = 0"
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        """Behavior policy: ε-greedy"""
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_rewards = []
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        state = env.reset()  # "Initialize S"
        total_reward = 0
        
        # "Loop for each step of episode:"
        while True:
            action = epsilon_greedy(state)  # "Choose A from S using policy derived from Q (e.g., ε-greedy)"
            next_state, reward, done = env.step(action)  # "Take action A, observe R, S'"
            total_reward += reward
            
            # "Q(S,A) ← Q(S,A) + α[R + γ max_a Q(S',a) − Q(S,A)]" - Equation 6.8
            td_target = reward + gamma * np.max(Q[next_state])
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:  # "until S is terminal"
                break
            
            state = next_state  # "S ← S'"
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_qlearning, rewards_qlearning = q_learning(env, n_episodes=500)
print(f"Q-Learning trained for {len(Q_qlearning)} states")

In [ ]:
# SARSA vs Q-Learning karşılaştırması
# Referans: Example 6.6 (s. 132) - "Cliff Walking"
# "The lower graph shows learning curves ... Q-learning learns values for the 
# optimal policy... Sarsa, on the other hand, takes the action-Loss into account 
# and learns the longer but safer path through the upper part of the grid."

def compare_algorithms(env, n_runs=10, n_episodes=500):
    """
    SARSA ve Q-Learning'i karşılaştır.
    Referans: Figure 6.4 (s. 132)
    """
    
    sarsa_rewards = np.zeros((n_runs, n_episodes))
    qlearn_rewards = np.zeros((n_runs, n_episodes))
    
    for run in range(n_runs):
        _, rewards_s = sarsa(env, n_episodes=n_episodes)
        _, rewards_q = q_learning(env, n_episodes=n_episodes)
        
        sarsa_rewards[run] = rewards_s
        qlearn_rewards[run] = rewards_q
    
    return sarsa_rewards.mean(axis=0), qlearn_rewards.mean(axis=0)

sarsa_avg, qlearn_avg = compare_algorithms(env, n_runs=10, n_episodes=500)

# Plot - Figure 6.4 (s. 132)
plt.figure(figsize=(12, 5))

window = 10
sarsa_smooth = np.convolve(sarsa_avg, np.ones(window)/window, mode='valid')
qlearn_smooth = np.convolve(qlearn_avg, np.ones(window)/window, mode='valid')

plt.plot(sarsa_smooth, label='SARSA', color='blue')
plt.plot(qlearn_smooth, label='Q-Learning', color='red')
plt.xlabel('Episode')
plt.ylabel('Sum of Rewards during Episode')
plt.title('SARSA vs Q-Learning on Cliff Walking - Figure 6.4 (s. 132)')
plt.legend()
plt.ylim(-100, 0)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def extract_path(env, Q):
    """Q'dan greedy policy ile path çıkar."""
    path = []
    state = env.reset()
    path.append(state)
    
    for _ in range(100):  # Max steps
        action = np.argmax(Q[state])
        next_state, _, done = env.step(action)
        path.append(next_state)
        
        if done:
            break
        state = next_state
    
    return path

# Visualize learned paths
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# SARSA path
ax = axes[0]
path_sarsa = extract_path(env, Q_sarsa)
for row in range(env.rows):
    for col in range(env.cols):
        pos = (row, col)
        if pos == env.start:
            color = 'lightgreen'
        elif pos == env.goal:
            color = 'gold'
        elif pos in env.cliff:
            color = 'gray'
        else:
            color = 'white'
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)

path_x = [p[1] + 0.5 for p in path_sarsa]
path_y = [env.rows - p[0] - 0.5 for p in path_sarsa]
ax.plot(path_x, path_y, 'b-o', linewidth=2, markersize=6)
ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SARSA: Safe Path', fontsize=14)

# Q-Learning path
ax = axes[1]
path_qlearn = extract_path(env, Q_qlearning)
for row in range(env.rows):
    for col in range(env.cols):
        pos = (row, col)
        if pos == env.start:
            color = 'lightgreen'
        elif pos == env.goal:
            color = 'gold'
        elif pos in env.cliff:
            color = 'gray'
        else:
            color = 'white'
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)

path_x = [p[1] + 0.5 for p in path_qlearn]
path_y = [env.rows - p[0] - 0.5 for p in path_qlearn]
ax.plot(path_x, path_y, 'r-o', linewidth=2, markersize=6)
ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Q-Learning: Optimal Path', fontsize=14)

plt.tight_layout()
plt.show()

---
## 6.5 Expected SARSA

📖 **Referans:** Sutton & Barto, Sayfa 133-134, Section 6.6

> *"Consider the learning algorithm that is just like Q-learning except that instead of the maximum over next state–action pairs it uses the expected value, taking into account how likely each action is under the current policy."* (s. 133)

Q-Learning ve SARSA'nın ortası - Equation 6.9 (s. 133):

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \sum_a \pi(a|S_{t+1}) Q(S_{t+1}, a) - Q(S_t, A_t)]$$

### Avantajları (s. 133-134)
> *"Expected Sarsa is more complex computationally than Sarsa but, in return, it eliminates the variance due to the random selection of $A_{t+1}$."*

- SARSA'dan daha düşük variance
- Hem on-policy hem off-policy olabilir

In [ ]:
def expected_sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Expected SARSA.
    Referans: Equation 6.9 (s. 133)
    
    "uses the expected value, taking into account how likely each action 
    is under the current policy"
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    def expected_value(state):
        """
        E[Q(s,a)] under ε-greedy policy.
        "taking into account how likely each action is under the current policy"
        """
        q_values = Q[state]
        best_action = np.argmax(q_values)
        
        # ε-greedy action probabilities
        policy_probs = np.ones(env.n_actions) * epsilon / env.n_actions
        policy_probs[best_action] += 1 - epsilon
        
        # "Σ_a π(a|S_{t+1}) Q(S_{t+1}, a)"
        return np.dot(policy_probs, q_values)
    
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        while True:
            action = epsilon_greedy(state)
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # Expected SARSA update - Equation 6.9
            # "Q(S,A) ← Q(S,A) + α[R + γ E_π[Q(S',·)] − Q(S,A)]"
            td_target = reward + gamma * expected_value(next_state)
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards

Q_expected, rewards_expected = expected_sarsa(env, n_episodes=500)
print(f"Expected SARSA trained")

---
## 6.6 Maximization Bias ve Double Q-Learning

📖 **Referans:** Sutton & Barto, Sayfa 134-136, Section 6.7

### Problem: Maximization Bias (s. 134)

> *"All the control algorithms that we have discussed so far involve maximization in the construction of their target policies... In these algorithms, a maximum over estimated values is used implicitly as an estimate of the maximum value, which can lead to a significant positive bias."* (s. 134)

Q-Learning'de $\max_a Q(s', a)$ kullanımı **pozitif bias** yaratır.

Neden? $E[\max(X_1, X_2, ...)] \geq \max(E[X_1], E[X_2], ...)$

### Çözüm: Double Q-Learning (s. 135-136)

> *"The idea of double learning... divide the plays in two sets and use them to learn two independent estimates... then use one estimate to determine the maximizing action and the other to provide the estimate of its value."* (s. 135)

İki Q fonksiyonu kullan:
- $Q_1$: Action seçimi için
- $Q_2$: Value tahmini için (veya tersi)

$$Q_1(S, A) \leftarrow Q_1(S, A) + \alpha [R + \gamma Q_2(S', \arg\max_a Q_1(S', a)) - Q_1(S, A)]$$

In [ ]:
def double_q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    Double Q-Learning.
    Referans: Algorithm (s. 136) - "Double Q-learning"
    """
    # "Initialize Q1(s, a) and Q2(s, a), for all s ∈ S+, a ∈ A(s), such that Q(terminal, ·) = 0"
    Q1 = defaultdict(lambda: np.zeros(env.n_actions))
    Q2 = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        # "Choose A from S using the policy ε-greedy in Q1 + Q2"
        return np.argmax(Q1[state] + Q2[state])
    
    episode_rewards = []
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        state = env.reset()  # "Initialize S"
        total_reward = 0
        
        # "Loop for each step of episode:"
        while True:
            action = epsilon_greedy(state)  # "Choose A from S using... ε-greedy in Q1 + Q2"
            next_state, reward, done = env.step(action)  # "Take action A, observe R, S'"
            total_reward += reward
            
            # "With 0.5 probability:"
            if np.random.random() < 0.5:
                # "Q1(S,A) ← Q1(S,A) + α(R + γQ2(S', argmax_a Q1(S',a)) − Q1(S,A))"
                best_action = np.argmax(Q1[next_state])
                td_target = reward + gamma * Q2[next_state][best_action]
                Q1[state][action] += alpha * (td_target - Q1[state][action])
            else:
                # "else: Q2(S,A) ← Q2(S,A) + α(R + γQ1(S', argmax_a Q2(S',a)) − Q2(S,A))"
                best_action = np.argmax(Q2[next_state])
                td_target = reward + gamma * Q1[next_state][best_action]
                Q2[state][action] += alpha * (td_target - Q2[state][action])
            
            if done:  # "until S is terminal"
                break
            
            state = next_state  # "S ← S'"
        
        episode_rewards.append(total_reward)
    
    # Combine Q1 and Q2
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    all_states = set(Q1.keys()) | set(Q2.keys())
    for s in all_states:
        Q[s] = (Q1[s] + Q2[s]) / 2
    
    return Q, episode_rewards

Q_double, rewards_double = double_q_learning(env, n_episodes=500)
print("Double Q-Learning trained")

In [ ]:
# Tüm algoritmaları karşılaştır
n_runs = 10
n_episodes = 500

algorithms = {
    'SARSA': sarsa,
    'Q-Learning': q_learning,
    'Expected SARSA': expected_sarsa,
    'Double Q-Learning': double_q_learning
}

results = {}
for name, algo in algorithms.items():
    all_rewards = []
    for _ in range(n_runs):
        _, rewards = algo(env, n_episodes=n_episodes)
        all_rewards.append(rewards)
    results[name] = np.mean(all_rewards, axis=0)

# Plot
plt.figure(figsize=(12, 6))
colors = ['blue', 'red', 'green', 'purple']
window = 10

for (name, rewards), color in zip(results.items(), colors):
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=name, color=color, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Sum of Rewards')
plt.title('TD Control Algorithms Comparison')
plt.legend()
plt.ylim(-100, 0)
plt.grid(True, alpha=0.3)
plt.show()

---
## Özet

📖 **Chapter 6 Key Points (s. 115-145)**

| Algoritma | Sayfa | Tip | Update Target |
|-----------|-------|-----|---------------|
| **TD(0)** | s. 119-120 | Prediction | $R + \gamma V(S')$ |
| **SARSA** | s. 129-131 | On-policy Control | $R + \gamma Q(S', A')$ |
| **Q-Learning** | s. 131-132 | Off-policy Control | $R + \gamma \max_a Q(S', a)$ |
| **Expected SARSA** | s. 133-134 | Both | $R + \gamma E_\pi[Q(S', a)]$ |
| **Double Q** | s. 135-136 | Off-policy Control | Decoupled selection & evaluation |

### SARSA vs Q-Learning (Example 6.6, s. 132)

| | SARSA | Q-Learning |
|--|-------|------------|
| Policy | On-policy | Off-policy |
| Öğrendiği | ε-greedy policy'nin değeri | Optimal policy'nin değeri |
| Davranış | Daha "güvenli" | Daha "riskli" |
| Cliff örneğinde | Uçurumdan uzak | Uçurum kenarından |

### Anahtar Denklemler

**TD Error** (Eq. 6.5, s. 121):
$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

**SARSA** (Eq. 6.7, s. 130):
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

**Q-Learning** (Eq. 6.8, s. 131):
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

---
### Sonraki Notebook
**07 - N-Step Bootstrapping** *(Chapter 7, s. 147-162)*: TD ve MC arasındaki spektrum